In [5]:
import json
import pandas as pd

def jprint(obj):
    print(json.dumps(obj, indent=2))


In [17]:
records = [
    {
        "file": "/home/baris/.cache/huggingface/hub/datasets--bdsaglam--predictions-ragent-Qwen2.5-1.5B-Instruct-musique-grpo/snapshots/eab98df035c11b20a8c84ec361debe955ac27d9b/predictions-ragent-Qwen2.5-1.5B-Instruct-musique-grpo.jsonl",
        "model": "Qwen2.5-1.5B-Instruct-ragent-grpo-musique",
        "retriever": "bm25",
    },
    {
        "file": "../outputs/musique-predictions-ragent-qwen-2.5-1.5B-Instruct.jsonl",
        "model": "Qwen2.5-1.5B-Instruct",
        "retriever": "hybrid",
    },
    {
        "file": "../outputs/ragent/meta-llama/Llama-3.1-8B-Instruct/predictions-musique-mini-hybrid.jsonl",
        "model": "Llama-3.1-8B-Instruct",
        "retriever": "hybrid",
    },
]

In [18]:
def load_predictions(record):
    with open(record['file']) as f:
        records = [json.loads(line) for line in f]
    df = pd.DataFrame(records)
    df['model'] = record['model']
    df['retriever'] = record['retriever']
    return df


def load_all_predictions(records):
    dfs = [load_predictions(record) for record in records]
    return pd.concat(dfs)

In [19]:
from verifiers.rubrics.utils import get_last_answer
from verifiers.metrics.musique import exact_match, f1

def preprocess(df):
    df['n_hops'] = df['supporting_titles'].apply(len)
    df['predicted_answer'] = df['trajectory'].apply(get_last_answer)
    df['exact_match'] = df.apply(lambda row: exact_match(row['predicted_answer'], row['answers']), axis=1)
    df['f1'] = df.apply(lambda row: f1(row['predicted_answer'], row['answers']), axis=1)
    return df

In [20]:
df = preprocess(load_all_predictions(records))
df

,answer,n_hops,prompt,docs,answers,supporting_titles,trajectory,model,retriever,id,predicted_answer,exact_match,f1
0,Vito Corleone,2,[{'content': 'Answer the question based on the...,"[{'id': 0, 'is_supporting': False, 'text': '# ...","[Vito Corleone, Vito Andolini, Vito Andolini C...","[The Good Shepherd (film), The Godfather Part II]",[{'content': 'Answer the question based on the...,Qwen2.5-1.5B-Instruct-ragent-grpo-musique,bm25,NaN,Robert De Niro,0,0.0
1,Rohana Wijeweera,2,[{'content': 'Answer the question based on the...,"[{'id': 0, 'is_supporting': False, 'text': '# ...",[Rohana Wijeweera],"[Rohana Wijeweera, Dimuthu Bandara Abayakoon]",[{'content': 'Answer the question based on the...,Qwen2.5-1.5B-Instruct-ragent-grpo-musique,bm25,NaN,Janatha Vimukthi Peramuna,0,0.0
2,406,2,[{'content': 'Answer the question based on the...,"[{'id': 0, 'is_supporting': False, 'text': '# ...",[406],"[Rössen culture, Galicia (Spain)]",[{'content': 'Answer the question based on the...,Qwen2.5-1.5B-Instruct-ragent-grpo-musique,bm25,NaN,1932,0,0.0
3,Jamie Murray,2,[{'content': 'Answer the question based on the...,"[{'id': 0, 'is_supporting': False, 'text': '# ...",[Jamie Murray],"[2011 Valencia Open 500 – Doubles, 2016 Wimble...",[{'content': 'Answer the question based on the...,Qwen2.5-1.5B-Instruct-ragent-grpo-musique,bm25,NaN,Roger Federer,0,0.0
4,modern-day Italians,2,[{'content': 'Answer the question based on the...,"[{'id': 0, 'is_supporting': False, 'text': '# ...",[modern-day Italians],"[Jews, Ashkenazi Jews]",[{'content': 'Answer the question based on the...,Qwen2.5-1.5B-Instruct-ragent-grpo-musique,bm25,NaN,Ashkenazi Jews,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,the first Sunday in April,4,[{'content': 'Answer the question based on the...,"[{'id': 0, 'is_supporting': False, 'text': '# ...",[the first Sunday in April],"[The Messenger (Zusak novel), 1952 Winter Olym...",[{'content': 'Answer the question based on the...,Llama-3.1-8B-Instruct,hybrid,4hop1__58323_375563_161848_83118,Croatia,0,0.0
296,2013,4,[{'content': 'Answer the question based on the...,"[{'id': 0, 'is_supporting': False, 'text': '# ...",[2013],"[Desde El Principio, Sony Music, Santa Monica,...",[{'content': 'Answer the question based on the...,Llama-3.1-8B-Instruct,hybrid,4hop1__151650_5274_458768_33677,"Santa Monica, California",0,0.0
297,Treaty of Paris,4,[{'content': 'Answer the question based on the...,"[{'id': 0, 'is_supporting': False, 'text': '# ...",[Treaty of Paris],"[Riverside Plaza, Southeast Library, Minneapol...",[{'content': 'Answer the question based on the...,Llama-3.1-8B-Instruct,hybrid,4hop1__94201_642284_131926_13165,Treaty of Paris,1,1.0
298,Mario Andretti,4,[{'content': 'Answer the question based on the...,"[{'id': 0, 'is_supporting': False, 'text': '# ...",[Mario Andretti],"[Charles Mingus, Tijuana Moods, Tucson, Arizon...",[{'content': 'Answer the question based on the...,Llama-3.1-8B-Instruct,hybrid,4hop1__726152_153080_33897_81096,Croatia,0,0.0


In [22]:
df.groupby(['model', 'retriever'])[['exact_match', 'f1']].agg(['count', 'mean', 'std'])

exact_match            \
                                                          count      mean   
model                                     retriever                         
Llama-3.1-8B-Instruct                     hybrid            300  0.213333   
Qwen2.5-1.5B-Instruct                     hybrid            300  0.003333   
Qwen2.5-1.5B-Instruct-ragent-grpo-musique bm25              300  0.086667   

                                                                 f1            \
                                                          std count      mean   
model                                     retriever                             
Llama-3.1-8B-Instruct                     hybrid     0.410346   300  0.250381   
Qwen2.5-1.5B-Instruct                     hybrid     0.057735   300  0.004667   
Qwen2.5-1.5B-Instruct-ragent-grpo-musique bm25       0.281816   300  0.138532   

                                                               
                                                          std  
model                                     retriever            
Llama-3.1-8B-Instruct                     hybrid     0.416848  
Qwen2.5-1.5B-Instruct                     hybrid     0.062111  
Qwen2.5-1.5B-Instruct-ragent-grpo-musique bm25       0.310839

In [12]:
import textwrap
from ipywidgets import widgets
from IPython.display import display
from ipywidgets import HBox
from tabulate import tabulate

def fixedwidth(text):
    """Wrap text to fixed width"""
    return "\n".join(textwrap.wrap(str(text), width=120, replace_whitespace=False))

def format_messages(messages):
    """Format a list of messages as a chat"""
    return "\n".join([f"{m['role']}: {m['content']}" for m in messages])

def format_row(row):
    """Format a single row for display as a table"""
    n_prefix_messages = len(row['prompt'])
    data = [
        ["F1", f"{row['f1']:.2f}"],
        ["Answers", row['answers']],
        ["Predicted Answer", row['predicted_answer']],
        ["Prompt", fixedwidth(format_messages(row['prompt'][-1:]))], 
        ["Trajectory", fixedwidth(format_messages(row['trajectory'][n_prefix_messages:]))],
    ]
    return tabulate(data, tablefmt='grid')

def present_row(row):
    """Display a formatted row as table"""
    print(format_row(row))

def create_browse_app(df):
    """Create an interactive widget to browse through the data"""
    def browse_data(i=0):
        row = df.iloc[i]
        present_row(row)

    index = widgets.IntText(value=0, description='Index:')
    left_button = widgets.Button(description='Previous')
    right_button = widgets.Button(description='Next')

    def on_left_button_clicked(b):
        if index.value > 0:
            index.value -= 1

    def on_right_button_clicked(b):
        if index.value < len(df) - 1:
            index.value += 1

    left_button.on_click(on_left_button_clicked)
    right_button.on_click(on_right_button_clicked)

    ui = HBox([left_button, index, right_button])
    out = widgets.interactive_output(browse_data, {'i': index})

    display(ui, out)

In [18]:
mask = df['n_hops'] == 2
sdf = df[mask]
create_browse_app(sdf)

Output()